In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer
import pickle

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_csv("heart_disease_uci.csv")

# -------------------------
# CLEANING
# -------------------------
df.replace('?', np.nan, inplace=True)

# Drop useless columns
df = df.drop(columns=["id", "dataset"], errors="ignore")

# -------------------------
# SAFE BINARY ENCODING
# -------------------------
def safe_binary(col):
    if col in df.columns:
        df[col] = df[col].replace({
            'Male': 1, 'Female': 0,
            'Yes': 1, 'No': 0,
            'TRUE': 1, 'FALSE': 0,
            'True': 1, 'False': 0
        })
        df[col] = pd.to_numeric(df[col], errors='coerce')

safe_binary('sex')
safe_binary('fbs')
safe_binary('exang')

# -------------------------
# MULTI-CLASS CATEGORICAL ENCODING
# -------------------------
for col in ['cp', 'restecg', 'slope', 'thal']:
    if col in df.columns:
        df[col] = df[col].astype('category').cat.codes

# -------------------------
# NUMERIC CONVERSION
# -------------------------
numeric_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# -------------------------
# TARGET COLUMN
# -------------------------
if 'target' in df.columns:
    target_col = 'target'
elif 'num' in df.columns:
    target_col = 'num'
    df[target_col] = df[target_col].apply(lambda x: 1 if x > 0 else 0)
else:
    raise Exception("No target column found!")

# -------------------------
# FEATURES / TARGET SPLIT
# -------------------------
X = df.drop(target_col, axis=1)
y = df[target_col]

# 🔥 Remove completely empty columns
X = X.dropna(axis=1, how='all')

feature_names = X.columns.tolist()

# -------------------------
# IMPUTER
# -------------------------
imputer = SimpleImputer(strategy='median')

X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=feature_names
)

# -------------------------
# TRAIN / TEST SPLIT
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------
# SCALING
# -------------------------
scaler = StandardScaler()

X_train = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_names
)

X_test = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_names
)

# -------------------------
# MODEL
# -------------------------
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

# -------------------------
# EVALUATION
# -------------------------
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# -------------------------
# SAVE FILES
# -------------------------
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(imputer, open("imputer.pkl", "wb"))
pickle.dump(feature_names, open("features.pkl", "wb"))

print("\n Model saved successfully!")

# -------------------------
# DOWNLOAD
# -------------------------
from google.colab import files

files.download("model.pkl")
files.download("scaler.pkl")
files.download("imputer.pkl")
files.download("features.pkl")


Accuracy: 0.7880434782608695

Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.84      0.76        75
           1       0.87      0.75      0.81       109

    accuracy                           0.79       184
   macro avg       0.79      0.80      0.79       184
weighted avg       0.80      0.79      0.79       184


 Model saved successfully!


/tmp/ipykernel_4285/3578447877.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace({


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python --version

Python 3.12.13
